# 淘宝用户行为分析

数据：阿里天池 UserBehavior，约 1 亿条行为日志，观测窗口 9 天（2017-11-25 ~ 12-03）。

分析口径：
- **用户级漏斗**：9 天内是否发生过该行为，衡量人群覆盖
- **会话级漏斗**：相邻行为间隔 > 30 分钟切分，衡量单次访问效率
- **时间**：统一 +8 小时转北京时间
- 所有查询同步维护在 `sql/` 目录，Notebook 只负责调用与展示


## 0. 环境准备


In [ ]:
import sys
!{sys.executable} -m pip install duckdb

In [ ]:
import os
from pathlib import Path

# notebook 位于 notebooks/ 时，切回项目根目录，保证相对路径 data/ 可用
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT)
print("项目根目录：", ROOT)
print("数据存在：", Path("data/UserBehavior.csv").exists())


## 1. 数据导入


In [ ]:
import duckdb, time

t0 = time.time()
con = duckdb.connect("taobao.duckdb")

con.execute("""
CREATE OR REPLACE TABLE ub AS
SELECT * FROM read_csv_auto('data/UserBehavior.csv', header=false,
  columns={'user_id':'BIGINT','item_id':'BIGINT','category_id':'BIGINT',
           'behavior_type':'VARCHAR','timestamp':'BIGINT'})
""")
print("rows:", con.execute("SELECT COUNT(*) FROM ub").fetchone()[0],
      f"| {time.time()-t0:.0f}s")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

rows: 100150807 | 5s


## 2. 数据校验

行数、主键基数、时间戳值域三件套。只看 `COUNT(*)` 发现不了时间戳异常。


In [ ]:
print(con.execute("""
SELECT COUNT(*) AS rows,
       COUNT(DISTINCT user_id) AS users,
       COUNT(DISTINCT item_id) AS items,
       MIN(timestamp) AS t0,
       MAX(timestamp) AS t1
FROM ub
""").df())

        rows   users    items          t0          t1
0  100150807  987994  4162024 -2134949234  2122867355


In [ ]:
print(con.execute("""
SELECT behavior_type, COUNT(*) AS cnt,
       ROUND(100.0*COUNT(*)/SUM(COUNT(*)) OVER (), 2) AS pct
FROM ub GROUP BY 1 ORDER BY cnt DESC
""").df())

  behavior_type       cnt    pct
0            pv  89716264  89.58
1          cart   5530446   5.52
2           fav   2888258   2.88
3           buy   2015839   2.01


## 3. 数据清洗

原始数据中约 5.5 万行（0.055%）落在观测窗口外，按左闭右开区间过滤。


In [ ]:
print(con.execute("""
SELECT CAST(make_timestamp(timestamp*1000000) + INTERVAL 8 HOUR AS DATE) AS dt,
       COUNT(*) AS events,
       COUNT(DISTINCT user_id) AS dau
FROM ub GROUP BY 1 ORDER BY 1
""").df())


In [ ]:
print(con.execute("""
SELECT COUNT(*) AS bad_rows,
       ROUND(100.0*COUNT(*)/(SELECT COUNT(*) FROM ub), 4) AS pct
FROM ub
WHERE timestamp < 1511539200 OR timestamp > 1512316800
""").df())

   bad_rows     pct
0     54751  0.0547


In [ ]:
con.execute("""
CREATE OR REPLACE VIEW ub_clean AS
SELECT * FROM ub
WHERE timestamp >= 1511539200 AND timestamp < 1512316800
""")
print(con.execute("SELECT COUNT(*) FROM ub_clean").fetchone())


## 4. 用户级漏斗

9 天内既浏览过又购买过的用户占比，衡量「人群覆盖」。


In [ ]:
print(con.execute("""
WITH u AS (
  SELECT user_id,
         MAX(CASE WHEN behavior_type='pv'   THEN 1 ELSE 0 END) AS has_pv,
         MAX(CASE WHEN behavior_type='cart' THEN 1 ELSE 0 END) AS has_cart,
         MAX(CASE WHEN behavior_type='fav'  THEN 1 ELSE 0 END) AS has_fav,
         MAX(CASE WHEN behavior_type='buy'  THEN 1 ELSE 0 END) AS has_buy
  FROM ub_clean
  GROUP BY 1
)
SELECT COUNT(*) AS users,
       SUM(has_pv)   AS pv_users,
       SUM(has_cart) AS cart_users,
       SUM(has_buy)  AS buy_users,
       ROUND(100.0*SUM(has_cart)/SUM(has_pv), 2) AS pv2cart,
       ROUND(100.0*SUM(has_buy) /SUM(has_pv), 2) AS pv2buy,
       ROUND(100.0*SUM(has_buy) /SUM(has_cart), 2) AS cart2buy
FROM u
""").df())

    users  pv_users  cart_users  buy_users  pv2cart  pv2buy  cart2buy
0  987991  984105.0    738996.0   672404.0    75.09   68.33     90.99


## 5. 品类分析

按品类看流量与转化的关系，最小样本阈值 5,000。


In [ ]:
print(con.execute("""
WITH c AS (
  SELECT category_id, user_id,
         MAX(CASE WHEN behavior_type='pv'  THEN 1 ELSE 0 END) AS pv_u,
         MAX(CASE WHEN behavior_type='buy' THEN 1 ELSE 0 END) AS buy_u
  FROM ub_clean
  GROUP BY 1, 2
)
SELECT category_id,
       SUM(pv_u) AS pv_users,
       SUM(buy_u) AS buy_users,
       ROUND(100.0*SUM(buy_u)/SUM(pv_u), 2) AS cvr,
       ROUND(100.0*SUM(pv_u)/(SELECT SUM(pv_u) FROM c), 3) AS pv_share
FROM c
GROUP BY 1
HAVING SUM(pv_u) >= 5000
ORDER BY pv_users DESC
LIMIT 20
""").df())

    category_id  pv_users  buy_users    cvr  pv_share
0       4756105  410413.0    23754.0   5.79     1.795
1        982926  392749.0    21190.0   5.40     1.718
2       4145813  368451.0    27529.0   7.47     1.612
3       2355072  361786.0    10523.0   2.91     1.583
4       1320293  343972.0    14903.0   4.33     1.505
5       3607361  331738.0    10623.0   3.20     1.451
6       4801426  321351.0    22725.0   7.07     1.406
7       2520377  297246.0     8141.0   2.74     1.300
8       2465336  262538.0    11263.0   4.29     1.148
9       3002561  255728.0    14669.0   5.74     1.119
10      2735466  255223.0    28759.0  11.27     1.116
11      1080785  226443.0     4027.0   1.78     0.991
12      4217906  201053.0    12462.0   6.20     0.879
13       149192  197030.0     9655.0   4.90     0.862
14      2520771  176131.0     9386.0   5.33     0.770
15      1464116  170789.0    29953.0  17.54     0.747
16      4181361  166701.0     7556.0   4.53     0.729
17      2920476  164824.0   

## 6. 时段分析

区分「量的峰」与「率的峰」，并拆分工作日 / 周末。


In [ ]:
print(con.execute("""
SELECT EXTRACT(HOUR FROM make_timestamp(timestamp*1000000) + INTERVAL 8 HOUR) AS hr,
       COUNT(*) AS events,
       COUNT(DISTINCT user_id) AS users,
       ROUND(100.0*SUM(CASE WHEN behavior_type='buy' THEN 1 ELSE 0 END)/COUNT(*), 2) AS buy_pct
FROM ub_clean
GROUP BY hr
ORDER BY hr
""").df())

    hr   events   users  buy_pct
0    0  3402700  318416     1.70
1    1  1581714  167421     1.46
2    2   855243   98337     1.40
3    3   583978   71419     1.37
4    4   499719   64219     1.35
5    5   646100   84446     1.26
6    6  1366543  179932     1.32
7    7  2488382  329996     1.51
8    8  3403022  444399     1.91
9    9  4182518  520529     2.30
10  10  4877472  577799     2.62
11  11  4741816  587527     2.57
12  12  4771071  593649     2.49
13  13  5208034  602301     2.37
14  14  5189893  593592     2.35
15  15  5365029  603739     2.29
16  16  5151948  598367     2.26
17  17  4697672  585509     2.16
18  18  4795879  583798     2.00
19  19  6021741  624683     1.91
20  20  7304809  663343     1.83
21  21  8369723  682751     1.74
22  22  8299813  649704     1.67
23  23  6291237  514021     1.59


In [ ]:
print(con.execute("""
SELECT EXTRACT(HOUR FROM make_timestamp(timestamp*1000000)+INTERVAL 8 HOUR) AS hr,
       CASE WHEN EXTRACT(ISODOW FROM make_timestamp(timestamp*1000000)+INTERVAL 8 HOUR) >= 6
            THEN '周末' ELSE '工作日' END AS day_type,
       COUNT(*) AS events,
       SUM(CASE WHEN behavior_type='buy' THEN 1 ELSE 0 END) AS buys
FROM ub_clean
GROUP BY 1, 2
ORDER BY 2, 1
""").df())

    hr day_type   events     buys
0    0       周末  1724236  25006.0
1    1       周末   821769  11241.0
2    2       周末   435966   5830.0
3    3       周末   287691   3814.0
4    4       周末   240849   3169.0
5    5       周末   299618   3519.0
6    6       周末   621758   7198.0
7    7       周末  1263253  17018.0
8    8       周末  1696561  29295.0
9    9       周末  2001755  42236.0
10  10       周末  2301848  57231.0
11  11       周末  2278249  56338.0
12  12       周末  2192381  52737.0
13  13       周末  2405120  55081.0
14  14       周末  2472996  56040.0
15  15       周末  2564879  56126.0
16  16       周末  2556655  54572.0
17  17       周末  2351119  48559.0
18  18       周末  2366285  45364.0
19  19       周末  2923948  53233.0
20  20       周末  3555104  61775.0
21  21       周末  4061869  67210.0
22  22       周末  4006786  63433.0
23  23       周末  3093950  46428.0
24   0      工作日  1678464  32770.0
25   1      工作日   759945  11928.0
26   2      工作日   419277   6182.0
27   3      工作日   296287   4212.0
28   4      工作

## 7. 回访率

9 天窗口不支持 cohort 留存，退化为逐日回访率，并排除最后一天。


In [ ]:
print(con.execute("""
WITH daily AS (
  SELECT DISTINCT user_id,
         CAST(make_timestamp(timestamp*1000000)+INTERVAL 8 HOUR AS DATE) AS d
  FROM ub_clean
  WHERE timestamp >= 1511539200 AND timestamp < 1512316800
)
SELECT a.d AS dt,
       COUNT(DISTINCT a.user_id) AS dau,
       COUNT(DISTINCT b.user_id) AS next_day_users,
       ROUND(100.0*COUNT(DISTINCT b.user_id)/COUNT(DISTINCT a.user_id), 2) AS d1_return
FROM daily a
LEFT JOIN daily b
  ON a.user_id = b.user_id AND b.d = a.d + 1
WHERE a.d < (SELECT MAX(d) FROM daily)
GROUP BY dt
ORDER BY dt
""").df())

          dt     dau  next_day_users  d1_return
0 2017-11-25  706641          557328      78.87
1 2017-11-26  715516          556179      77.73
2 2017-11-27  710094          557310      78.48
3 2017-11-28  709257          561269      79.13
4 2017-11-29  718922          571944      79.56
5 2017-11-30  730597          581754      79.63
6 2017-12-01  740139          727168      98.25
7 2017-12-02  970401          950784      97.98


## 8. 会话级漏斗（进阶）

用 `LAG` 计算行为间隔、累计 `SUM` 生成会话 ID，衡量单次访问的转化效率。


In [ ]:
print(con.execute("""
WITH ev AS (
  SELECT user_id, timestamp, behavior_type,
         COALESCE(timestamp - LAG(timestamp) OVER (
             PARTITION BY user_id ORDER BY timestamp), 999999) AS gap
  FROM ub_clean
),
marked AS (
  SELECT *, CASE WHEN gap > 1800 THEN 1 ELSE 0 END AS new_sess
  FROM ev
),
sess AS (
  SELECT user_id, behavior_type,
         SUM(new_sess) OVER (PARTITION BY user_id ORDER BY timestamp
                             ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS sid
  FROM marked
),
agg AS (
  SELECT user_id, sid,
         MAX(CASE WHEN behavior_type='pv'  THEN 1 ELSE 0 END) AS has_pv,
         MAX(CASE WHEN behavior_type='buy' THEN 1 ELSE 0 END) AS has_buy
  FROM sess GROUP BY 1, 2
)
SELECT COUNT(*) AS sessions,
       SUM(has_pv) AS pv_sessions,
       SUM(has_buy) AS buy_sessions,
       ROUND(100.0*SUM(has_buy)/SUM(has_pv), 2) AS pv2buy_session_pct
FROM agg
WHERE has_pv = 1
""").df())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   sessions  pv_sessions  buy_sessions  pv2buy_session_pct
0  15580093   15580093.0     1179194.0                7.57


## 9. RFM 分层

只对购买行为计算 R（最近购买）与 F（购买频次），各用 `NTILE(5)` 分成 1–5 分。

> 局限：9 天窗口下 R 的取值被截断在 0–9 天，分层仅作方法演示；`COUNT(*)` 数的是行为条数而非订单数。


In [ ]:
print(con.execute("""
WITH base AS (
  SELECT user_id,
         MAX(CAST(make_timestamp(timestamp*1000000)+INTERVAL 8 HOUR AS DATE)) AS last_buy,
         COUNT(*) AS f
  FROM ub_clean
  WHERE behavior_type = 'buy'
  GROUP BY user_id
),
scored AS (
  SELECT user_id, last_buy, f,
         DATE '2017-12-04' - last_buy                    AS r_days,
         NTILE(5) OVER (ORDER BY last_buy)               AS r_score,  -- 越近分越高
         NTILE(5) OVER (ORDER BY f)                      AS f_score   -- 越多分越高
  FROM base
)
SELECT user_id, last_buy, r_days, f, r_score, f_score,
       r_score + f_score AS rf_score
FROM scored
ORDER BY rf_score DESC, r_days
LIMIT 10
""").df())

## 10. 主要结论

- **口径差 9 倍**：用户级转化率 68.33%，会话级 7.57%。前者衡量人群覆盖，后者衡量单次访问效率，不可混用。
- **量与率错位**：事件量峰值在 21 点（837 万），购买占比峰值在 10 点（2.62%）；晚间流量多 72%，但单量只多 14%。
- **流量与转化无关**：品类转化率从 1.39% 到 18.28%，相差 13 倍，而浏览份额集中在 0.6%~1.8%。
